# Coleta ArXiv — Poda estrutural e compressão de LLMs

Orquestra os módulos em `src/collection/` e `src/preprocessing/`.
Tema: structural pruning / LLM compression (2023–2026, cs.CL/cs.LG/cs.AI).

In [4]:
import sys
from pathlib import Path

ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

import json
import pandas as pd

from src.collection.config import CollectionConfig
from src.collection.query_builder import build_arxiv_query
from src.collection.fetcher import fetch_arxiv_raw
from src.collection.normalizer import normalize_corpus
from src.preprocessing.config import PreprocessConfig
from src.preprocessing.pipeline import preprocess_document, preprocess_query

## 1. Configuração da coleta

In [5]:
config = CollectionConfig(
    theme="Structural pruning and LLM compression during fine-tuning",
    keywords=[
        "structural pruning",
        "LLM compression",
        "attention heads pruning",
        "MLP pruning",
        "parameter efficient fine-tuning",
    ],
    categories=["cs.CL", "cs.LG", "cs.AI"],
    year_from=2023,
    year_to=2026,
    target_count=2000,
    page_size=50,
)
config.validate()

query = build_arxiv_query(config.keywords, config.categories)
print("Query:", query)
print("Target:", config.target_count)

Query: (all:"structural pruning" OR all:"LLM compression" OR all:"attention heads pruning" OR all:"MLP pruning" OR all:"parameter efficient fine-tuning") AND (cat:cs.CL OR cat:cs.LG OR cat:cs.AI)
Target: 2000


## 2. Coleta (idempotente, retomável)

In [6]:
n_raw = fetch_arxiv_raw(config)
print(f"Raw unique ids: {n_raw}")

collecting:   0%|          | 0/2000 [00:00<?, ?it/s]


[warning] collection interrupted (attempt 1/6): HTTPError: Page request resulted in HTTP 429 (https://export.arxiv.org/api/query?search_query=%28all%3A%22structural+pruning%22+OR+all%3A%22LLM+compression%22+OR+all%3A%22attention+heads+pruning%22+OR+all%3A%22MLP+pruning%22+OR+all%3A%22parameter+efficient+fine-tuning%22%29+AND+%28cat%3Acs.CL+OR+cat%3Acs.LG+OR+cat%3Acs.AI%29&id_list=&sortBy=submittedDate&sortOrder=descending&start=0&max_results=50)
[warning] waiting 60s before resuming from offset=0...




backoff:   0%|          | 0/60 [00:00<?, ?it/s]

backoff:   2%|▏         | 1/60 [00:01<00:59,  1.00s/it]

backoff:   3%|▎         | 2/60 [00:02<00:58,  1.00s/it]

backoff:   5%|▌         | 3/60 [00:03<00:57,  1.00s/it]

backoff:   7%|▋         | 4/60 [00:04<00:56,  1.00s/it]

backoff:   8%|▊         | 5/60 [00:05<00:55,  1.00s/it]

backoff:  10%|█         | 6/60 [00:06<00:54,  1.00s/it]

backoff:  12%|█▏        | 7/60 [00:07<00:53,  1.00s/it]

backoff:  13%|█▎        | 8/60 [00:08<00:52,  1.00s/it]

backoff:  15%|█▌        | 9/60 [00:09<00:51,  1.00s/it]

backoff:  17%|█▋        | 10/60 [00:10<00:50,  1.00s/it]

backoff:  18%|█▊        | 11/60 [00:11<00:49,  1.01s/it]

backoff:  20%|██        | 12/60 [00:12<00:48,  1.00s/it]

backoff:  22%|██▏       | 13/60 [00:13<00:47,  1.00s/it]

backoff:  23%|██▎       | 14/60 [00:14<00:46,  1.00s/it]

backoff:  25%|██▌       | 15/60 [00:15<00:45,  1.00s/it]

backoff:  27%|██▋       | 16/60 [00:16<00:44,  1.00s/it]

backoff:  28%|██▊       | 17/6


[warning] collection interrupted (attempt 2/6): HTTPError: Page request resulted in HTTP 429 (https://export.arxiv.org/api/query?search_query=%28all%3A%22structural+pruning%22+OR+all%3A%22LLM+compression%22+OR+all%3A%22attention+heads+pruning%22+OR+all%3A%22MLP+pruning%22+OR+all%3A%22parameter+efficient+fine-tuning%22%29+AND+%28cat%3Acs.CL+OR+cat%3Acs.LG+OR+cat%3Acs.AI%29&id_list=&sortBy=submittedDate&sortOrder=descending&start=50&max_results=50)
[warning] waiting 120s before resuming from offset=50...


collecting:   2%|▎         | 50/2000 [16:02<10:25:56, 19.26s/it, offset=50]



[warning] collection interrupted (attempt 3/6): HTTPError: Page request resulted in HTTP 429 (https://export.arxiv.org/api/query?search_query=%28all%3A%22structural+pruning%22+OR+all%3A%22LLM+compression%22+OR+all%3A%22attention+heads+pruning%22+OR+all%3A%22MLP+pruning%22+OR+all%3A%22parameter+efficient+fine-tuning%22%29+AND+%28cat%3Acs.CL+OR+cat%3Acs.LG+OR+cat%3Acs.AI%29&id_list=&sortBy=submittedDate&sortOrder=descending&start=50&max_results=50)
[warning] waiting 240s before resuming from offset=50...




backoff:   0%|          | 0/240 [00:00<?, ?it/s]

backoff:   0%|          | 1/240 [00:01<03:59,  1.00s/it]

backoff:   1%|          | 2/240 [00:02<03:58,  1.00s/it]

backoff:   1%|▏         | 3/240 [00:03<03:57,  1.00s/it]

backoff:   2%|▏         | 4/240 [00:04<03:56,  1.00s/it]

backoff:   2%|▏         | 5/240 [00:05<03:55,  1.00s/it]

backoff:   2%|▎         | 6/240 [00:06<03:54,  1.00s/it]

backoff:   3%|▎         | 7/240 [00:07<03:53,  1.00s/it]

backoff:   3%|▎         | 8/240 [00:08<03:52,  1.00s/it]

backoff:   4%|▍         | 9/240 [00:09<03:51,  1.00s/it]

backoff:   4%|▍         | 10/240 [00:10<03:50,  1.00s/it]

backoff:   5%|▍         | 11/240 [00:11<03:49,  1.00s/it]

backoff:   5%|▌         | 12/240 [00:12<03:48,  1.00s/it]

backoff:   5%|▌         | 13/240 [00:13<03:47,  1.00s/it]

backoff:   6%|▌         | 14/240 [00:14<03:46,  1.00s/it]

backoff:   6%|▋         | 15/240 [00:15<03:45,  1.00s/it]

backoff:   7%|▋         | 16/240 [00:16<03:44,  1.00s/it]

backoff:   7%


[warning] collection interrupted (attempt 4/6): HTTPError: Page request resulted in HTTP 503 (https://export.arxiv.org/api/query?search_query=%28all%3A%22structural+pruning%22+OR+all%3A%22LLM+compression%22+OR+all%3A%22attention+heads+pruning%22+OR+all%3A%22MLP+pruning%22+OR+all%3A%22parameter+efficient+fine-tuning%22%29+AND+%28cat%3Acs.CL+OR+cat%3Acs.LG+OR+cat%3Acs.AI%29&id_list=&sortBy=submittedDate&sortOrder=descending&start=50&max_results=50)
[warning] waiting 480s before resuming from offset=50...


collecting:  84%|████████▍ | 1681/2000 [03:43<00:43,  7.31it/s, offset=1681]

Raw unique ids: 1681


## 3. Normalização → corpus.jsonl

In [ ]:
n_corpus = normalize_corpus(config)
print(f"Corpus documents: {n_corpus}")

## 4. Sanity checks

In [ ]:
records = [json.loads(l) for l in config.corpus_path.read_text().strip().splitlines()]
df = pd.DataFrame(records)
df["year"] = pd.to_datetime(df["date"]).dt.year
print("Count:", len(df))
print("\nBy year:")
print(df["year"].value_counts().sort_index())
print("\nBy primary category:")
print(df["categories"].apply(lambda c: c[0] if c else "").value_counts().head(10))

## 5. Demonstração de pré-processamento

In [ ]:
pp_cfg = PreprocessConfig(use_stemming=False)

sample_query = "structural pruning of attention heads in large language models"
query_tokens = preprocess_query(sample_query, pp_cfg)
print("Query tokens:", query_tokens[:20])

sample_doc = records[0]
doc_tokens = preprocess_document(sample_doc, pp_cfg)
print("Doc tokens:", doc_tokens[:20])

pp_stem = PreprocessConfig(use_stemming=True)
stem_tokens = preprocess_query(sample_query, pp_stem)
print("With stemming:", stem_tokens[:20])